# MonteCarlo Off-Policy

*Description*: En este notebook se desarrolla la implementación del método de **SARSA semi-gradiente**, y se emplea sobre el entorno MountainCar de Gymnasium.


    Autores: David Fernández Expósito
             Ángel Martínez Sánchez
             Javier Polo Gambín

    Emails: dfernandezexposito@um.es
            angel.martinezs@um.es
            javier.polog@um.es
            
    Date: 2026/02/20


Empezamos instalando e importando las librerías necesarias. También definimos los dispositivos donde se ejecutará el notebook y la semilla que vamos a usar para asegurar reproducibilidad.

In [1]:
! pip install "gymnasium[classic_control]"

In [2]:
# Importamos todas las clases y funciones
import sys

# Añadir los directorio fuentes al path de Python
sys.path.append('./src')


# Verificar que se han añadido correctamente
print(sys.path)

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import gymnasium as gym
import torch
import gc
import os
from tiling.tiles3 import tiles, IHT

['C:\\Users\\angel\\AppData\\Local\\Programs\\Python\\Python311\\python311.zip', 'C:\\Users\\angel\\AppData\\Local\\Programs\\Python\\Python311\\DLLs', 'C:\\Users\\angel\\AppData\\Local\\Programs\\Python\\Python311\\Lib', 'C:\\Users\\angel\\AppData\\Local\\Programs\\Python\\Python311', 'c:\\Users\\angel\\Desktop\\Master-Practicas-Repos\\FernandezMartinezPolo-EML-RL\\.venv', '', 'c:\\Users\\angel\\Desktop\\Master-Practicas-Repos\\FernandezMartinezPolo-EML-RL\\.venv\\Lib\\site-packages', './src']


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")


gc.collect()              # Ejecuta el recolector de basura de Python
if torch.cuda.is_available():
    torch.cuda.empty_cache()   # Limpia la caché de la GPU


SEED = 123

# NumPy
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Python
os.environ["PYTHONHASHSEED"] = str(SEED)

Usando dispositivo: cpu


Antes de implementar nada es fundamental explorar el entorno para saber exactamente con qué estamos trabajando.

In [ ]:
# Se crea el entorno MountainCar
env = gym.make("MountainCar-v0", render_mode="human")

# Se resetea el estado usando la semilla
env.reset(seed=SEED)

# Se explora la estructura del espacio de acciones
print("Espacio de acciones:")
print(f"\t- Tipo: {env.action_space}")
print(f"\t- Número de acciones: {env.action_space.n}")
print()

# Se exploran los límites inferiorer y superiores
# del espacio de estados (no se puede explorar la
# estructura de forma completa ya que MountainCar
# es un entorno continuo)
print("Espacio de estados:")
print(f"\t- Tipo: {env.observation_space}")
print(f"\t- Límites inferiores: {env.observation_space.low}")
print(f"\t- Límites superiores: {env.observation_space.high}")

# Se renderiza el estado por pantalla
env.render()

Espacio de acciones:
	- Tipo: Discrete(3)
	- Número de acciones: 3

Espacio de estados:
	- Tipo: Box([-1.2  -0.07], [0.6  0.07], (2,), float32)
	- Límites inferiores: [-1.2  -0.07]
	- Límites superiores: [0.6  0.07]


: 

Esta salida nos dice lo siguiente:

- El espacio de acciones es `Discrete(3)`, es decir, hay exactamente 3 acciones posibles, codificadas como enteros:
  - 0: empujar hacia la izquierda.
  - 1: no hacer nada.
  - 2: empujar hacia la derecha.
  
- El espacio de estados es continuo y bidimensional:
  - La primera dimensión es la posición del coche, que va desde -1,2 hasta 0,6.
  - La segunda dimensión es la velocidad, que va desde -0,07 hasta 0,07.
  
A continuación, se lleva a cabo un episodio para comprobar cómo funciona el entorno.

In [5]:
# Hacemos un episodio aleatorio para ver cómo funciona
state, info = env.reset()

print("Estado inicial:")
print(f"\t- Estado: {state}")
print(f"\t- Posición: {state[0]:.4f}")
print(f"\t- Velocidad: {state[1]:.4f}")
print()

# Se ejecutan los 3 primeros pasos tomando acciones
# completamente aleatorias
print("Primeros 3 pasos:")
for i in range(3):
    # Se elige una acción cualquiera del espacio
    # de acciones
    action = env.action_space.sample()

    # Se toma la acción y se recoge información sobre
    # el estado siguiente, la recompensa, si el nuevo
    # estado es terminal, si el episodio se ha truncado
    # por haber alcanzado el límite de pasos (200 por
    # defecto en MountainCar) e información adicional
    next_state, reward, terminated, truncated, info = env.step(action)

    # Se imprime por pantalla información sobre el
    # nuevo estado
    print(f"\t- Paso {i+1}:")
    print(f"\t\t- Acción: {action}")
    print(f"\t\t- Recompensa: {reward}")
    print(f"\t\t- Posición: {next_state[0]:.4f}")
    print(f"\t\t- Velocidad: {next_state[1]:.4f}")
    print(f"\t\t- Terminado: {terminated}")

# Se cierra el entorno ya que hemos dejado de
# usarlo (al menos por ahora)
env.close()

Estado inicial:
	- Estado: [-0.5892358  0.       ]
	- Posición: -0.5892
	- Velocidad: 0.0000

Primeros 3 pasos:
	- Paso 1:
		- Acción: 1
		- Recompensa: -1.0
		- Posición: -0.5887
		- Velocidad: 0.0005
		- Terminado: False
	- Paso 2:
		- Acción: 0
		- Recompensa: -1.0
		- Posición: -0.5888
		- Velocidad: -0.0000
		- Terminado: False
	- Paso 3:
		- Acción: 1
		- Recompensa: -1.0
		- Posición: -0.5883
		- Velocidad: 0.0005
		- Terminado: False


Esta salida nos dice lo siguiente:

- El estado inicial comienza en la posición -0,41 en el eje X con una velocidad nula (el coche se encuentra en un valle).

- En los tres pasos dados se eligen acciones aleatorias (izquierda, derecha, derecha, en ese orden). Terminado es `False` en las tres ya que el coche no ha llegado a la meta, y la recompensa es -1 en cada paso ya que el coche es penalizado con una recompensa de -1 por cada paso que tarda sin alcanzar la cima. Esto incentiva al agente a alcanzar la cima lo más rápido posible.

Antes de pasar a 